In [ ]:
import pandas as pd 
from pathlib import Path
import os
import sys, pathlib
pandas as pd, json, re, asyncio
from helpers.config import get_settings

sys.path.insert(0, str(pathlib.Path.cwd().parent))  # adjust if the notebook is deeper

In [2]:

settings = get_settings()
current_path = Path.cwd().resolve().parent

data_path = os.path.join(current_path, 'src/assets/files/electronics_cellphones/uvcdhebaa7jv_electronics_cellphones.csv')
df  = pd.read_csv(data_path)

df = df.dropna(subset=["title"]).drop_duplicates("parent_asin").reset_index(drop=True)
df["description"] = df["description"].fillna("")
df["text"] = df["title"] + "\n" + df["description"]
print(len(df))

12730


In [3]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),       # catches "screen protector", "apple watch"
    min_df=3,                 # drop rare tokens (model codes, typos)
    max_df=0.4,               # drop boilerplate in many titles ("compatible", "pack")
    sublinear_tf=True,
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b",   # letters only, ignores numbers and sizes
)
X = tfidf.fit_transform(df["title"])

svd = TruncatedSVD(n_components=100, random_state=42)
emb = normalize(svd.fit_transform(X))      # dense, L2-normalized, ready for KMeans
print(X.shape, emb.shape)

for k in [10, 15, 20, 30]:
    labels = KMeans(k, random_state=42, n_init="auto").fit_predict(emb)
    print(k, round(silhouette_score(emb, labels), 3))

(12730, 18122) (12730, 100)
10 0.093
15 0.111
20 0.124
30 0.139


In [5]:
K = 35
km = KMeans(K, random_state=42, n_init="auto").fit(emb)
df["stratum"] = km.labels_

terms = np.array(tfidf.get_feature_names_out())
sizes = df["stratum"].value_counts().sort_values(ascending=False)
for c in sizes.index:
    centroid = np.asarray(X[km.labels_ == c].mean(axis=0)).ravel()
    top = ", ".join(terms[centroid.argsort()[::-1][:5]])
    print(f"{c:>2} ({sizes[c]:>4}): {top}")

28 (1206): power, fan, wireless, black, smart
17 ( 824): galaxy, samsung, samsung galaxy, case, ultra
13 ( 772): case, iphone, cover, phone, phone case
 6 ( 675): ssd, ram, intel, laptop, core
 1 ( 633): usb, cable, adapter, type, cord
 5 ( 566): charger, fast, iphone, charging, apple
 8 ( 559): pro max, max, iphone pro, pro, iphone
15 ( 527): camera, hdmi, video, monitor, dash
29 ( 439): protector, glass, tempered, tempered glass, screen protector
12 ( 383): wallet, leather, case, card, wallet case
16 ( 382): tablet, ipad, case, inch, generation
10 ( 370): earbuds, wireless, wireless earbuds, bluetooth, headphones
34 ( 368): bundle, kit, camera, lens, mount
25 ( 354): ssd, intel, ram, core, windows
14 ( 325): radio, player, car, fm, lcd
27 ( 318): phone, cell, cell phone, popsockets, popgrip
21 ( 295): headphones, ear, headset, bluetooth, wireless
33 ( 290): heavy duty, heavy, duty, case, shockproof
 2 ( 277): protector, screen, screen protector, film, anti
31 ( 256): battery, charger

In [6]:
mask = df["stratum"] == 28
sub = KMeans(8, random_state=42, n_init="auto").fit(emb[mask.values])

# give the new sub-clusters fresh ids so they don't collide with 0-34
df.loc[mask, "stratum"] = 100 + sub.labels_

sub_df = df[mask].copy()
for c in sorted(sub_df["stratum"].unique()):
    g = sub_df[sub_df.stratum == c]
    centroid = np.asarray(X[mask.values][sub.labels_ == c - 100].mean(axis=0)).ravel()
    top = ", ".join(terms[centroid.argsort()[::-1][:5]])
    print(f"{c} ({len(g)}): {top}")
    print(g["title"].sample(min(3, len(g)), random_state=0).str[:70].to_string(index=False))

100 (127): smart watch, watch, smart, fitness, tracker
BASICF Military Smart Watches for Men (Answer/M...
MICOKON Waterproof Fitness Watch with Dial/Rece...
Qonioi Smart Watch - 1.7Inch Full-Touch-Screen ...
101 (119): power, supply, power supply, ac, dc
Replacement 60W Power Adapter L-Tip Connector f...
Television Repair Kit for Samsung UN58TU7000FXZ...
J-ZMQER AC Adapter Compatible with X Hover-1 My...
102 (51): telescope, watching, bird watching, bird, binoculars
Fetch Falcon 10-30x50 High Power FMC HD Zoom Mo...
40x60 Monocular-Telescope Low Night Vision Mono...
VULNESS Binoculars for Adults & Kids - Compact ...
103 (151): laptop, black, stand, computer, laptop stand
K2 Notebook Bracket Folding Portable Laptop Hol...
Gaevenwin Large Photo Case 4"x6",Seed Craft Org...
Broonel - Profile Series - Black Leather Laptop...
104 (160): keyboard, mouse, wireless, pc, bluetooth
Wireless Keyboard and Mouse Combo, Compact Quie...
5.0 Bluetooth Adapter USB Bluetooth Transmitter...
Pack of 4 Mic

In [8]:
def stratified_sample(df, n=150, cap=8, seed=42):
    frac = df["stratum"].value_counts(normalize=True)
    alloc = (frac * n).round().clip(lower=1, upper=cap).astype(int)
    parts = [
        g.sample(min(alloc[s], len(g)), random_state=seed)
        for s, g in df.groupby("stratum")
    ]
    return pd.concat(parts).reset_index(drop=True)

sample = stratified_sample(df, n=150, cap=8)
print(len(sample), sample["stratum"].nunique())
print(sample["stratum"].value_counts().sort_index())

145 42
stratum
0      3
1      7
2      3
3      1
4      3
5      7
6      8
7      3
8      7
9      1
10     4
11     2
12     5
13     8
14     4
15     6
16     5
17     8
18     2
19     2
20     3
21     3
22     2
23     3
24     1
25     4
26     3
27     4
29     5
30     2
31     3
32     2
33     3
34     4
100    1
101    1
102    1
103    2
104    2
105    4
106    1
107    2
Name: count, dtype: int64


In [9]:
from langchain_cohere import ChatCohere

llm = ChatCohere(model=settings.GENERATION_MODEL, temperature=0.5, max_tokens=512)

PROMPT = """You are a shopper. For the product below, write 4 different search queries in English
that would each lead you to buy it:
1. "category": product type + brand or device model (3-6 words)
2. "feature": focused on one specific attribute (size, material, pack, color, capability)
3. "short": a vague 2-3 word query
4. "need": a natural sentence describing what you want to accomplish

Do not copy the title. Return only JSON: {{"category": "...", "feature": "...", "short": "...", "need": "..."}}

Product:
{text}"""

def parse_json(s):
    m = re.search(r"\{.*\}", s, re.S)
    return json.loads(m.group(0)) if m else {}

async def make_queries(sem, row, retries=3):
    async with sem:
        for attempt in range(retries):
            try:
                r = await llm.ainvoke(PROMPT.format(text=row["text"][:1200]))
                q = parse_json(r.content)
                return [{"query": v.strip(), "angle": k,
                         "product_id": row["parent_asin"], "stratum": int(row["stratum"])}
                        for k, v in q.items() if isinstance(v, str) and v.strip()]
            except Exception as e:
                await asyncio.sleep(2 ** attempt * 3)   # backoff for rate limits / bad JSON
        return []

async def build(sample, workers=3):
    sem = asyncio.Semaphore(workers)
    out = await asyncio.gather(*[make_queries(sem, r) for _, r in sample.iterrows()])
    return pd.DataFrame([x for rows in out for x in rows])

evalset = await build(sample)
print(len(evalset), evalset["product_id"].nunique())

508 127


In [11]:
evalset.to_csv("evalset.csv", index=False)